# Red Cedar — Full Pipeline Example

This notebook demonstrates the complete QML workflow using the Red Cedar Python client:
1. Connect to the server
2. Select a dataset
3. Run PCA analysis
4. Run Mutual Information analysis
5. Run Quantum Resource Estimation
6. Train a model
7. Run inference (classification)
8. Run Explainable AI

All computation happens on the server. This notebook only sends requests and displays results.

In [ ]:
import redcedar

# Connect to the server
# Reads REDCEDAR_URL and REDCEDAR_API_KEY from environment, or pass explicitly:
# client = redcedar.connect(url="http://localhost:8000", api_key="your-key")
client = redcedar.connect()
print(f"Red Cedar SDK v{redcedar.__version__}")
print(f"Server: {client.url}")
print(f"Status: {client.status()}")

## 1. List Available Datasets

In [ ]:
datasets = client.datasets.list()
print(f"Available datasets ({len(datasets)}):")
for ds in datasets:
    print(f"  {ds['id']:20s} {ds['name']:30s} {ds.get('rows', '?')} rows, {ds.get('classes', '?')} classes")

In [ ]:
# Select a dataset for this demo
dataset = "wdbc"
print(f"Selected: {dataset}")

## 2. PCA Analysis

Principal Component Analysis reveals the intrinsic dimensionality of the data — how many independent dimensions carry real signal.

In [ ]:
pca = client.analysis.pca(dataset, verbose=True)

print(f"\nDataset: {pca['dataset_name']}")
print(f"Features: {pca['features']}")
print(f"Samples: {pca['samples']}")
print(f"Components for 90% variance: {pca['thresholds']['90']}")
print(f"Components for 95% variance: {pca['thresholds']['95']}")
print(f"Elapsed: {pca.get('elapsed_seconds', '?')}s")

## 3. Mutual Information Analysis

MI ranks features by how much they predict the class label. The quantum model will use the top-ranked features.

In [ ]:
mi = client.analysis.mi(dataset, verbose=True)

print(f"\nTotal MI: {mi['total_mi']:.4f}")
print(f"Features for 90% MI: {mi['thresholds']['90']}")
print(f"\nTop 10 features:")
for f in mi['ranked_features'][:10]:
    print(f"  {f['rank']+1:2d}. {f['feature_name']:25s} MI={f['mi_score']:.4f} ({f['mi_percent']:.1f}%)")

## 4. Quantum Resource Estimation

Estimates the number of qubits needed for the quantum model to achieve high accuracy on this dataset.

In [ ]:
qre = client.analysis.qre(dataset, verbose=True)

print(f"\nEstimated qubits: {qre['estimatedQubits']} +/- {qre['standardDeviation']:.1f}")
print(f"Samples: {qre['samples']}, Features: {qre['features']}, Classes: {qre['classes']}")
print(f"Elapsed: {qre.get('elapsed_seconds', '?')}s")

## 5. Model Training

Train a quantum machine learning model. Settings can be overridden — here we use 4 qubits with 1 layer as a quick demo.

In [ ]:
result = client.training.run(dataset, verbose=True, settings={
    "num_qubits": 4,
    "min_layers": 1,
    "max_layers": 1,
})

train_id = result['train_id']
print(f"\nModel ID: {train_id}")
print(f"Elapsed: {result.get('elapsed_seconds', '?')}s")

## 6. Inference (Classification)

Test the trained model on held-out test data.

In [ ]:
inference = client.inference.run(train_id, verbose=True)

print(f"\nOverall accuracy: {inference['overall_accuracy']:.1%}")
print(f"Overall loss: {inference['overall_loss']:.4f}")
print(f"Test samples: {inference['test_samples']}")
print(f"\nPer-class results:")
for cls in inference['per_class']:
    print(f"  Class {cls['class']}: accuracy={cls['accuracy']:.1%}, samples={cls['samples']}")

## 7. Explainable AI

Identify which input features drive the model's classification decisions.

In [ ]:
explanation = client.explainable.run(train_id, verbose=True)

print(f"\nDataset: {explanation['dataset_name']}")
print(f"Qubits: {explanation['num_qubits']} ({explanation['num_class_qubits']} class, {explanation['num_qubits'] - explanation['num_class_qubits']} data)")

if explanation.get('qubit_map'):
    print(f"\nKey features (data qubit -> feature):")
    for q in explanation['qubit_map']:
        print(f"  Qubit {q['qubit']}: {q['feature_name']} (col {q['feature_index']}, importance {q['importance']*100:.1f}%)")

print(f"\nConvergence:")
for qid, val in explanation.get('convergence', {}).items():
    print(f"  Class qubit {qid}: {val:.4f}")

## Summary

This notebook demonstrated the complete Red Cedar pipeline:
1. **PCA** revealed the dataset's intrinsic dimensionality
2. **MI** identified the most predictive features
3. **QRE** estimated the qubits needed
4. **Training** built a quantum model
5. **Inference** tested it on held-out data
6. **Explainable AI** showed which features drive decisions

All computation ran on the server. This notebook required only `pip install redcedar`.